In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

backend = BasicSimulator()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 65.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=db1c28fbfcee883234d4abfaf53f3b05d4a8b78ba3320b5e55b710698a21ebd2
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

## Overview

BB84 with an eavesdropper, Eve, performing **intercept-and-resend**: she catches every qubit Alice sends, measures it in a random basis, then forwards a fresh qubit to Bob encoding her measured bit in her basis.

Alice and Bob proceed normally — they don't know Eve is there. The point of the simulation is that the public error-check sample reveals a disturbance Eve cannot avoid creating.

All random choices come from quantum measurements of $H|0\rangle$. Sections are labelled [ALICE], [EVE], [BOB], [CLASSICAL].

Protocol steps:
1. Alice picks random bits and bases
2. Eve picks random interception bases
3. Bob picks random measurement bases
4. Alice encodes → Eve intercepts and resends → Bob measures
5. Sifting on Alice's and Bob's bases (Eve's bases stay hidden)
6. Error check on a sample of sifted bits
7. Breakdown of where Eve's disturbance came from

## Quantum random bit generator

Prepare $|+\rangle$, measure in Z, get 0 or 1 with probability $\tfrac{1}{2}$. Shared by all three parties.

In [3]:
def quantum_random_bits(n: int) -> list[int]:
    """Return n random bits from measuring n independent |+> qubits."""
    out = []
    for _ in range(n):
        qc = QuantumCircuit(1, 1)
        qc.h(0)
        qc.measure(0, 0)
        tqc = transpile(qc, backend)
        counts = backend.run(tqc, shots=1).result().get_counts()
        out.append(int(next(iter(counts))))
    return out

## Alice — encoding

Same encoding as in the plain protocol.

| Bit | Z basis (0) | X basis (1) |
|-----|-------------|-------------|
| 0   | $\lvert 0\rangle$ | $\lvert +\rangle$ |
| 1   | $\lvert 1\rangle$ | $\lvert -\rangle$ |

In [4]:
def alice_encode(bit: int, basis: int) -> QuantumCircuit:
    """Build the qubit Alice sends for (bit, basis). No measurement."""
    qc = QuantumCircuit(1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

## Eve — intercept and resend

For every qubit:
1. Eve picks a basis (Z or X).
2. She measures Alice's qubit in that basis. This collapses the state and yields a bit.
3. She prepares a fresh qubit encoding that bit in her basis and forwards it to Bob.

She cannot clone the qubit (no-cloning theorem), so step 2 is unavoidable. Whenever her basis disagrees with Alice's, her measurement irreversibly randomises the state, which is what shows up in the error check downstream.

In [5]:
def eve_attack(received: QuantumCircuit, basis: int) -> tuple[int, QuantumCircuit]:
    """Measure the incoming qubit in `basis`, return (measured_bit, fresh_qubit_for_bob)."""
    # Measure Alice's qubit
    qc = QuantumCircuit(1, 1)
    qc.compose(received, inplace=True)
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    tqc = transpile(qc, backend)
    counts = backend.run(tqc, shots=1).result().get_counts()
    measured = int(next(iter(counts)))

    # Re-encode and forward to Bob
    resent = alice_encode(measured, basis)
    return measured, resent

## Bob — measurement

Bob measures the qubit he receives. He doesn't know Eve has tampered with it.

In [6]:
def bob_measure(received: QuantumCircuit, basis: int) -> int:
    """Measure the incoming qubit in Bob's chosen basis; return 0 or 1."""
    qc = QuantumCircuit(1, 1)
    qc.compose(received, inplace=True)
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    tqc = transpile(qc, backend)
    counts = backend.run(tqc, shots=1).result().get_counts()
    return int(next(iter(counts)))

## Run the protocol

In [7]:
N         = 100    # qubits transmitted
THRESHOLD = 0.15   # abort if observed error rate exceeds this
                   # no attacker: expected ~0%, with Eve full-interception: expected ~25%

print("=" * 56)
print("  BB84 — with attacker (Eve, intercept-and-resend)")
print("=" * 56)
print(f"N = {N}, abort threshold = {THRESHOLD:.0%}\n")

  BB84 — with attacker (Eve, intercept-and-resend)
N = 100, abort threshold = 15%



In [8]:
# Step 1 — Alice's bits and bases
print("[ALICE] picking bits and bases")
alice_bits  = quantum_random_bits(N)
alice_bases = quantum_random_bits(N)

print(f"[ALICE]  bits  (first 20): {alice_bits[:20]}")
print(f"[ALICE]  bases (first 20): {alice_bases[:20]}   (0=Z, 1=X)")

[ALICE] picking bits and bases
[ALICE]  bits  (first 20): [0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0]
[ALICE]  bases (first 20): [1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1]   (0=Z, 1=X)


In [9]:
# Step 2 — Eve's interception bases
print("[EVE] picking interception bases")
eve_bases = quantum_random_bits(N)

print(f"[EVE]    bases (first 20): {eve_bases[:20]}   (0=Z, 1=X)")

[EVE] picking interception bases
[EVE]    bases (first 20): [0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1]   (0=Z, 1=X)


In [10]:
# Step 3 — Bob's measurement bases
print("[BOB] picking measurement bases")
bob_bases = quantum_random_bits(N)

print(f"[BOB]    bases (first 20): {bob_bases[:20]}   (0=Z, 1=X)")

[BOB] picking measurement bases
[BOB]    bases (first 20): [1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0]   (0=Z, 1=X)


In [11]:
# Step 4 — Quantum channel with Eve sitting in the middle
print("[CHANNEL] Alice encodes -> Eve intercepts + resends -> Bob measures")

eve_bits = []   # what Eve measured (her private info)
bob_bits = []   # what Bob measured

for i in range(N):
    sent_by_alice           = alice_encode(alice_bits[i], alice_bases[i])
    eve_measured, forwarded = eve_attack(sent_by_alice, eve_bases[i])
    eve_bits.append(eve_measured)
    bob_bits.append(bob_measure(forwarded, bob_bases[i]))

print(f"[EVE]    measured (first 20): {eve_bits[:20]}")
print(f"[BOB]    measured (first 20): {bob_bits[:20]}")

[CHANNEL] Alice encodes -> Eve intercepts + resends -> Bob measures
[EVE]    measured (first 20): [0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0]
[BOB]    measured (first 20): [1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1]


In [12]:
# Step 5 — Sifting (Alice's bases vs Bob's bases; Eve's are unknown to them)
print("[CLASSICAL] Alice and Bob compare bases, drop the mismatches")

keep     = [i for i in range(N) if alice_bases[i] == bob_bases[i]]
sifted_a = [alice_bits[i] for i in keep]
sifted_b = [bob_bits[i]   for i in keep]

print(f"[SIFT]   kept {len(keep)} / {N} positions")
print(f"[ALICE]  sifted (first 20): {sifted_a[:20]}")
print(f"[BOB]    sifted (first 20): {sifted_b[:20]}")

[CLASSICAL] Alice and Bob compare bases, drop the mismatches
[SIFT]   kept 52 / 100 positions
[ALICE]  sifted (first 20): [0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1]
[BOB]    sifted (first 20): [1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0]


In [13]:
# Step 6 — Error check on a sample of the sifted key
print("[CLASSICAL] revealing a sample to estimate the error rate")

sample_n = max(1, len(sifted_a) // 3)
errs     = sum(1 for a, b in zip(sifted_a[:sample_n], sifted_b[:sample_n]) if a != b)
err_rate = errs / sample_n

print(f"\n[CHECK]  sample size : {sample_n}")
print(f"[CHECK]  mismatches  : {errs}")
print(f"[CHECK]  error rate  : {err_rate:.2%}")
print(f"[CHECK]  threshold   : {THRESHOLD:.2%}")
print(f"[CHECK]  theory      : ~25% under full intercept-and-resend")

if err_rate > THRESHOLD:
    print(f"\n[ALERT] {err_rate:.2%} > {THRESHOLD:.2%} — eavesdropper detected, aborting")
else:
    print("\n[OK]    error rate within threshold (this is unusual for Eve — small sample noise)")

[CLASSICAL] revealing a sample to estimate the error rate

[CHECK]  sample size : 17
[CHECK]  mismatches  : 5
[CHECK]  error rate  : 29.41%
[CHECK]  threshold   : 15.00%
[CHECK]  theory      : ~25% under full intercept-and-resend

[ALERT] 29.41% > 15.00% — eavesdropper detected, aborting


## Where Eve's errors come from

Look at the sifted positions and split them by whether Eve happened to pick Alice's basis. When Eve got the basis right she learns Alice's bit and forwards a state Bob recovers correctly — no error. When Eve got it wrong her measurement randomises the qubit, and Bob's measurement (in Alice's basis) disagrees with Alice's bit half the time.

Expected split:
- Eve's basis = Alice's basis (50%) → 0% error
- Eve's basis ≠ Alice's basis (50%) → 50% error

Net error rate on the sifted key: 25%.

In [14]:
good_idx = [j for j, i in enumerate(keep) if eve_bases[i] == alice_bases[i]]   # Eve guessed right
bad_idx  = [j for j, i in enumerate(keep) if eve_bases[i] != alice_bases[i]]   # Eve guessed wrong

good_errs = sum(1 for j in good_idx if sifted_a[j] != sifted_b[j])
bad_errs  = sum(1 for j in bad_idx  if sifted_a[j] != sifted_b[j])

print("-" * 56)
print("Eve basis vs Alice basis breakdown (on sifted positions)")
print("-" * 56)
print(f"Eve guessed right : {len(good_idx):3d} positions, {good_errs:2d} errors"
      f"  ({0 if not good_idx else good_errs/len(good_idx):.1%})")
print(f"Eve guessed wrong : {len(bad_idx):3d} positions, {bad_errs:2d} errors"
      f"  ({0 if not bad_idx else bad_errs/len(bad_idx):.1%})")
print()
full_err = (good_errs + bad_errs) / len(keep) if keep else 0
print(f"Overall sifted-key error rate: {full_err:.2%}  (theory: ~25%)")

--------------------------------------------------------
Eve basis vs Alice basis breakdown (on sifted positions)
--------------------------------------------------------
Eve guessed right :  24 positions,  0 errors  (0.0%)
Eve guessed wrong :  28 positions, 15 errors  (53.6%)

Overall sifted-key error rate: 28.85%  (theory: ~25%)


## With Eve vs without

For comparison, run the protocol again with no attacker on the same `N`. Expect ~0% error vs ~25% with Eve.

In [15]:
def run_no_attacker(n):
    a_bits  = quantum_random_bits(n)
    a_bases = quantum_random_bits(n)
    b_bases = quantum_random_bits(n)
    b_bits  = [bob_measure(alice_encode(a_bits[i], a_bases[i]), b_bases[i]) for i in range(n)]
    k       = [i for i in range(n) if a_bases[i] == b_bases[i]]
    sa      = [a_bits[i] for i in k]
    sb      = [b_bits[i] for i in k]
    s       = max(1, len(sa) // 3)
    e       = sum(1 for x, y in zip(sa[:s], sb[:s]) if x != y)
    return len(sa), e / s

plain_sifted, plain_rate = run_no_attacker(N)

print("-" * 56)
print("Comparison")
print("-" * 56)
print(f"{'scenario':<20} {'sifted':>8} {'error rate':>12} {'detected':>12}")
print("-" * 56)
print(f"{'no attacker':<20} {plain_sifted:>8} {plain_rate:>12.2%} "
      f"{('yes' if plain_rate > THRESHOLD else 'no'):>12}")
print(f"{'with Eve':<20} {len(keep):>8} {err_rate:>12.2%} "
      f"{('yes' if err_rate > THRESHOLD else 'no'):>12}")
print(f"\nthreshold: {THRESHOLD:.2%}")

--------------------------------------------------------
Comparison
--------------------------------------------------------
scenario               sifted   error rate     detected
--------------------------------------------------------
no attacker                52        0.00%           no
with Eve                   52       29.41%          yes

threshold: 15.00%


## Summary

| Party | Role | What they do |
|-------|------|--------------|
| Alice | sender | encode random bits in random bases |
| Eve   | attacker | intercept, measure in random basis, resend a fresh qubit |
| Bob   | receiver | measure in random bases |

Eve's intercept-and-resend introduces a sifted-key error rate of about 25%, because:
- 50% of the time her basis matches Alice's → no disturbance,
- 50% of the time it doesn't → 50% chance of error for Bob.

$\tfrac{1}{2} \cdot 0 + \tfrac{1}{2} \cdot \tfrac{1}{2} = \tfrac{1}{4}$

Setting the threshold at 15% comfortably separates the two cases: a clean run sits near 0% (pass), Eve sits near 25% (abort). Single runs do fluctuate around those values, so the threshold has to leave enough margin for that noise.

The security argument is just the no-cloning theorem: Eve can't copy the qubit, so she has to measure, and measurement in the wrong basis is exactly what produces the disturbance that gives her away.